In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SujetA-AvisClients") \
    .config("spark.mongodb.write.connection.uri", "mongodb://sujetA-mongo:27017/sujetA") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

In [3]:
df = spark.read.csv("/home/jovyan/work/data/Reviews.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)
print("Nombre de lignes :", df.count())

root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = true)
 |-- HelpfulnessNumerator: string (nullable = true)
 |-- HelpfulnessDenominator: string (nullable = true)
 |-- Score: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)

+---+----------+--------------+--------------------+--------------------+----------------------+-----+----------+--------------------+--------------------+
| Id| ProductId|        UserId|         ProfileName|HelpfulnessNumerator|HelpfulnessDenominator|Score|      Time|             Summary|                Text|
+---+----------+--------------+--------------------+--------------------+----------------------+-----+----------+--------------------+--------------------+
|  1|B001E4KFG0|A3SGXH7AUHU8GW|          delmartian|                   1|                     1|    5|1303862400|Go

In [4]:
from pyspark.sql.functions import col, from_unixtime, to_date

df_clean = df.dropna(subset=["Score", "Text", "ProductId"]) \
    .withColumn("ReviewDate", to_date(from_unixtime(col("Time")))) \
    .drop("Time") \
    .dropDuplicates(["Id"])

df_clean.show(5)
print("Nombre de lignes après nettoyage :", df_clean.count())

+---+----------+--------------+--------------------+--------------------+----------------------+-----+--------------------+--------------------+----------+
| Id| ProductId|        UserId|         ProfileName|HelpfulnessNumerator|HelpfulnessDenominator|Score|             Summary|                Text|ReviewDate|
+---+----------+--------------+--------------------+--------------------+----------------------+-----+--------------------+--------------------+----------+
|  1|B001E4KFG0|A3SGXH7AUHU8GW|          delmartian|                   1|                     1|    5|Good Quality Dog ...|I have bought sev...|2011-04-27|
|  3|B000LQOCH0| ABXLMWJIXXAIN|"Natalia Corres "...|                   1|                     1|    4|"""Delight"" says...|"This is a confec...|2008-08-18|
|  6|B006K2ZZ7K| ADT0SRK1MGOEU|      Twoapennything|                   0|                     0|    4|          Nice Taffy|I got a wild hair...|2012-07-12|
| 12|B0009XLVG0|A2725IB4YY9JEB|"A Poeng ""Sparky...|            

In [5]:
from pyspark.sql.functions import count, avg, round as spark_round

# Note moyenne et nombre d'avis par produit
avis_par_produit = df_clean.groupBy("ProductId") \
    .agg(
        count("*").alias("nb_avis"),
        spark_round(avg("Score"), 2).alias("note_moyenne")
    ) \
    .orderBy(col("nb_avis").desc())

avis_par_produit.show(10)

# Répartition des notes (1 à 5)
repartition_notes = df_clean.groupBy("Score").count().orderBy("Score")
repartition_notes.show()

# Taux d'utilité des avis (HelpfulnessNumerator / HelpfulnessDenominator)
from pyspark.sql.functions import when

df_utilite = df_clean.withColumn(
    "taux_utilite",
    when(col("HelpfulnessDenominator") > 0,
         spark_round(col("HelpfulnessNumerator") / col("HelpfulnessDenominator"), 2))
    .otherwise(None)
)
df_utilite.select("Id", "ProductId", "HelpfulnessNumerator", "HelpfulnessDenominator", "taux_utilite").show(5)

+----------+-------+------------+
| ProductId|nb_avis|note_moyenne|
+----------+-------+------------+
|B007JFMH8M|    913|        4.58|
|B002QWHJOU|    632|        4.59|
|B002QWP89S|    632|        4.59|
|B0026RQTGE|    632|        4.59|
|B002QWP8H0|    632|        4.59|
|B003B3OOPA|    623|        4.78|
|B001EO5Q64|    567|        4.75|
|B007M832YY|    564|         4.3|
|B001RVFERK|    564|         4.3|
|B0026KNQSA|    564|         4.3|
+----------+-------+------------+
only showing top 10 rows

+--------------+-----+
|         Score|count|
+--------------+-----+
|          ..."|   23|
|     Author"""|    3|
|   Comp sci"""|    1|
|     Critic"""|    1|
|        Dad"""|    1|
|     Dance..."|    4|
|        Design|    1|
|        Ed..."|    1|
|       Hugs"""|    1|
|     Lyme ..."|    1|
|     Medit..."|    2|
|        Moscow|    1|
| Music Fan..."|    8|
|         RN"""|   23|
|        Sm..."|    1|
|      USA ..."|    1|
|   Video Games|    1|
|         a..."|   10|
|     and F..."

In [ ]:
df_clean.write \
    .format("mongodb") \
    .mode("overwrite") \
    .option("database", "sujetA") \
    .option("collection", "avis_nettoyes") \
    .save()

avis_par_produit.write \
    .format("mongodb") \
    .mode("overwrite") \
    .option("database", "sujetA") \
    .option("collection", "avis_par_produit") \
    .save()

repartition_notes.write \
    .format("mongodb") \
    .mode("overwrite") \
    .option("database", "sujetA") \
    .option("collection", "repartition_notes") \
    .save()